# 🐄 Taurus Vision — GPU Video Processor (v3 TUZATILGAN)

## ✅ Bu versiyada tuzatilganlar:
- **Embedding model TUZATILDI**: Endi backend bilan bir xil (ImageNet pretrained MobileNetV2). `mobilenet_muzzle.pt` KERAK EMAS!
- **Tracking TUZATILDI**: Sigir boshqa sigir orqasiga o'tsa (occlusion) — track saqlanadi, yangi track ochilmaydi
- **Re-ID qo'shildi**: Track vaqtincha yo'qolsa, embedding orqali qayta topiladi
- **ID_THRESHOLD pasaytirildi**: 0.80 → 0.70 (realistik chegara)

## ⚠️ Noutbukda avval bajaring:
```bash
# Terminal 1:
cd ~/taurus-vision && docker-compose up -d
# Terminal 2:
ngrok http 8000
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — GPU tekshirish + kutubxonalar
# ═══════════════════════════════════════════════════════════════════
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ GPU topildi: {r.stdout.strip()}')
else:
    print('❌ GPU topilmadi! Runtime → Change runtime type → T4 GPU')

import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'scipy', 'requests'], check=True)
print('✅ Kutubxonalar tayyor')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — SOZLAMALAR  ← BU YERNI TO'LDIRING
# ═══════════════════════════════════════════════════════════════════

# ngrok URL (oxirida '/' bo'lmasin)
BACKEND_URL = 'https://YOUR-NGROK-URL.ngrok-free.app'  # ← O'ZGARTIRING!

# Xavfsizlik kaliti
COLAB_SECRET = 'taurus123'

# Kamera ID
CAMERA_ID = 'COLAB-VIDEO-01'

# Identifikatsiya chegarasi
# MUHIM: 0.80 juda yuqori edi. 0.70 ko'proq sigirni taniydi.
# Agar noto'g'ri tanisa — 0.75 qiling. Umuman tanitmasa — 0.65.
ID_THRESHOLD = 0.70

# Har nechta kadrda bir DB ga yuborish
PUSH_EVERY_N = 3

# Chiqish video saqlash
SAVE_OUTPUT_VIDEO = True
OUTPUT_VIDEO_FILE = '/content/output_with_boxes.mp4'

# ─── URL tekshiruvi ───────────────────────────────────────────────
if 'YOUR-NGROK-URL' in BACKEND_URL:
    print('❌ BACKEND_URL hali o\'zgartirilmagan!')
else:
    import requests
    try:
        h = {'ngrok-skip-browser-warning': '1'}
        if COLAB_SECRET: h['X-Colab-Key'] = COLAB_SECRET
        r = requests.get(f'{BACKEND_URL}/health/ready', headers=h, timeout=10)
        if r.status_code == 200:
            print(f'✅ Backend ulandi: {BACKEND_URL}')
        else:
            print(f'⚠️ Status: {r.status_code} — docker-compose up -d ?')
    except Exception as e:
        print(f'❌ Aloqa yo\'q: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Model yuklash
# ═══════════════════════════════════════════════════════════════════
# YANGILIK: mobilenet_muzzle.pt KERAK EMAS!
# Backend bilan bir xil arxitektura ishlatiladi (ImageNet pretrained).
# Faqat best.pt (muzzle detector YOLO modeli) kerak.

from google.colab import files
import os

# Google Drive orqali (tavsiya — tezroq):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/best.pt' /content/best.pt

if not os.path.exists('/content/best.pt'):
    print('best.pt faylini tanlang (muzzle detector):')
    files.upload()
else:
    print('✅ best.pt allaqachon bor')

for f in os.listdir('/content/'):
    if f.endswith('.pt'):
        sz = os.path.getsize(f'/content/{f}')/1024/1024
        print(f'  {f}  ({sz:.1f} MB)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Embedding vektorlarini noutbukdan yuklash
# ═══════════════════════════════════════════════════════════════════
import requests, io, numpy as np

HEADERS = {'ngrok-skip-browser-warning': '1'}
if COLAB_SECRET:
    HEADERS['X-Colab-Key'] = COLAB_SECRET

print('Embeddinglar yuklanmoqda...')
r = requests.get(f'{BACKEND_URL}/api/v1/colab/export-embeddings',
                 headers=HEADERS, timeout=30)

if r.status_code != 200:
    print(f'❌ Xato {r.status_code}: {r.text[:300]}')
    print('  Jonivorlarni avval registration orqali ro\'yxatdan o\'tkazing!')
else:
    data = np.load(io.BytesIO(r.content))
    EMBEDDINGS  = data['embeddings']   # (N, 1280)
    ANIMAL_IDS  = data['animal_ids']   # (N,)
    TAG_IDS_RAW = data['tag_ids']      # (N,)

    # Har jonivor uchun o'rtacha L2-normalized embedding
    unique_ids     = np.unique(ANIMAL_IDS)
    emb_dim        = EMBEDDINGS.shape[1]
    AVG_EMBEDDINGS = np.zeros((len(unique_ids), emb_dim), dtype=np.float32)
    AVG_ANIMAL_IDS = []
    AVG_TAG_IDS    = []

    for i, aid in enumerate(unique_ids):
        mask = ANIMAL_IDS == aid
        v = EMBEDDINGS[mask].mean(axis=0)
        v = v / (np.linalg.norm(v) + 1e-8)
        AVG_EMBEDDINGS[i] = v
        AVG_ANIMAL_IDS.append(int(aid))
        AVG_TAG_IDS.append(str(TAG_IDS_RAW[mask][0]))

    AVG_ANIMAL_IDS = np.array(AVG_ANIMAL_IDS)

    print(f'✅ {len(EMBEDDINGS)} embedding, {len(unique_ids)} jonivor')
    print(f'   Embedding o\'lchami: {emb_dim}-dim (1280 bo\'lishi kerak)')
    print(f'   Teglar: {AVG_TAG_IDS[:10]}')

    if emb_dim != 1280:
        print('⚠️ OGOHLANTIRISH: O\'lcham 1280 emas — backend modelini tekshiring!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Video yuklash
# ═══════════════════════════════════════════════════════════════════
from google.colab import files
import os, cv2

print('Video faylni tanlang (mp4, avi, mov):')
uploaded = files.upload()
VIDEO_FILE = list(uploaded.keys())[0]

cap = cv2.VideoCapture(VIDEO_FILE)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps   = cap.get(cv2.CAP_PROP_FPS) or 25
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
size_mb = os.path.getsize(VIDEO_FILE) / 1024 / 1024

print(f'✅ Video: {VIDEO_FILE} ({size_mb:.1f} MB)')
print(f'   {total} kadr | {fps:.1f} FPS | {w}×{h}')
print(f'   Taxminiy vaqt (T4): ~{total//90:.0f} daqiqa')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — ASOSIY GPU PIPELINE (TUZATILGAN)
# ═══════════════════════════════════════════════════════════════════
import cv2, torch, numpy as np, requests, time, os
import torchvision.transforms as T
import torchvision.models as tvm
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from IPython.display import clear_output, display, Image as IPImage
import io as _io

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Modellar ────────────────────────────────────────────────────────
print('Modellar yuklanmoqda...')
yolo_model   = YOLO('yolo11n.pt').to(DEVICE)
muzzle_model = YOLO('best.pt').to(DEVICE)

# ── TUZATILGAN: Backend bilan bir xil MobileNetV2 ──────────────────
# Backend kodi (feature_extractor.py):
#   mobilenet.features + AdaptiveAvgPool2d(1,1) + Flatten → 1280-dim
#   weights = MobileNet_V2_Weights.IMAGENET1K_V1
# Colab ham xuddi shunday qilishi kerak!
_base = tvm.mobilenet_v2(weights=tvm.MobileNet_V2_Weights.IMAGENET1K_V1)
mobilenet = torch.nn.Sequential(
    _base.features,
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
)
for p in mobilenet.parameters():
    p.requires_grad = False
mobilenet.eval().to(DEVICE)
print('✅ MobileNetV2 (ImageNet pretrained, backend bilan mos) yuklandi')

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Ranglar (BGR) ───────────────────────────────────────────────────
COLOR = {
    'identified':   (0, 200, 80),
    'unidentified': (0, 60, 220),
    'tentative':    (0, 165, 255),
}

# ── Embedding chiqarish ─────────────────────────────────────────────
def get_embedding(crop_bgr):
    """Rasm dan 1280-dim L2-normalized embedding olish."""
    if crop_bgr is None or crop_bgr.size == 0:
        return None
    try:
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        inp = transform(rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            emb = mobilenet(inp).squeeze().cpu().numpy().astype(np.float32)
        emb /= (np.linalg.norm(emb) + 1e-8)
        return emb
    except:
        return None

# ── Identifikatsiya ─────────────────────────────────────────────────
def identify(emb):
    """Embedding bo'yicha jonivor topish."""
    if emb is None or len(AVG_EMBEDDINGS) == 0:
        return None, 0.0
    sims   = AVG_EMBEDDINGS @ emb
    best_i = int(np.argmax(sims))
    score  = float(sims[best_i])
    if score >= ID_THRESHOLD:
        return int(AVG_ANIMAL_IDS[best_i]), score
    return None, score

# ── Muzzle crop olish ───────────────────────────────────────────────
def get_muzzle_crop(frame, bbox_norm):
    """Sigir bbox dan muzzle qismini YOLO bilan kesib olish."""
    H, W = frame.shape[:2]
    cx, cy, bw, bh = bbox_norm
    x1 = max(0, int((cx - bw/2) * W))
    y1 = max(0, int((cy - bh/2) * H))
    x2 = min(W, int((cx + bw/2) * W))
    y2 = min(H, int((cy + bh/2) * H))
    body_crop = frame[y1:y2, x1:x2]
    if body_crop.size < 200:
        return body_crop
    # Muzzle detector
    mres = muzzle_model(body_crop, verbose=False, conf=0.25)
    if mres and len(mres[0].boxes) > 0:
        mb = mres[0].boxes.xywhn[0].cpu().numpy()
        mh, mw = body_crop.shape[:2]
        mx1 = max(0, int((mb[0] - mb[2]/2) * mw))
        my1 = max(0, int((mb[1] - mb[3]/2) * mh))
        mx2 = min(mw, int((mb[0] + mb[2]/2) * mw))
        my2 = min(mh, int((mb[1] + mb[3]/2) * mh))
        muzzle = body_crop[my1:my2, mx1:mx2]
        if muzzle.size > 200:
            return muzzle
    return body_crop

# ══════════════════════════════════════════════════════════════════════
# TUZATILGAN TRACKER — Occlusion bardoshli
# ══════════════════════════════════════════════════════════════════════
class Track:
    _counter = 0

    def __init__(self, bbox, conf):
        Track._counter += 1
        self.id       = Track._counter
        self.bbox     = bbox          # (cx, cy, w, h) normalized
        self.conf     = conf
        self.animal_id = None
        self.tag_id   = None
        self.id_score = 0.0
        self.hits     = 1
        self.age      = 0             # Ko'rinmay o'tgan kadrlar soni
        self.id_attempts = 0
        self.state    = 'tentative'
        self.last_emb = None          # ← YANGI: oxirgi embedding (Re-ID uchun)
        # Tezlik taxmini (bbox o'zgarishi)
        self.vel_cx   = 0.0
        self.vel_cy   = 0.0

    def predict_bbox(self):
        """Tezlik asosida keyingi pozitsiyani taxmin qilish."""
        cx, cy, bw, bh = self.bbox
        return (cx + self.vel_cx * self.age,
                cy + self.vel_cy * self.age,
                bw, bh)

    def update_velocity(self, new_bbox):
        cx_old, cy_old = self.bbox[0], self.bbox[1]
        cx_new, cy_new = new_bbox[0], new_bbox[1]
        alpha = 0.4  # Exponential smoothing
        self.vel_cx = alpha * (cx_new - cx_old) + (1 - alpha) * self.vel_cx
        self.vel_cy = alpha * (cy_new - cy_old) + (1 - alpha) * self.vel_cy


def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    xi = max(ax - aw/2, bx - bw/2)
    yi = max(ay - ah/2, by - bh/2)
    xa = min(ax + aw/2, bx + bw/2)
    ya = min(ay + ah/2, by + bh/2)
    inter = max(0, xa - xi) * max(0, ya - yi)
    return inter / (aw*ah + bw*bh - inter + 1e-8)


tracks = []

# Tracker parametrlari
IOU_MATCH_THRESHOLD = 0.15   # Eski: 0.25 (juda yuqori edi)
HITS_TO_CONFIRM     = 3      # Nechta kadrdan keyin 'unidentified'
MAX_AGE_FRAMES      = 45     # Eski: 20 (juda past edi — 1.5 soniya 30fps da)
ID_ATTEMPT_INTERVAL = 6      # Har nechta kadrda identification qilish
MAX_ID_ATTEMPTS     = 30     # Maksimum urinish
REID_THRESHOLD      = 0.75   # Re-ID cosine similarity chegarasi


def step_tracker(dets, frame, fno):
    """
    Tracker yangilanishi.
    dets: [(bbox_norm, conf), ...]
    """
    global tracks
    H, W = frame.shape[:2]

    # ── 1. IoU matching (predicted bbox bilan) ──────────────────────
    matched_t = set()
    matched_d = set()

    if tracks and dets:
        # Predicted bbox larni ishlatish (occlusion bardoshi uchun)
        pred_bboxes = [t.predict_bbox() for t in tracks]
        M = np.array([[iou(pb, d[0]) for d in dets] for pb in pred_bboxes])
        ri, ci = linear_sum_assignment(-M)
        for r, c in zip(ri, ci):
            if M[r, c] >= IOU_MATCH_THRESHOLD:
                t = tracks[r]
                t.update_velocity(dets[c][0])
                t.bbox  = dets[c][0]
                t.conf  = dets[c][1]
                t.hits += 1
                t.age   = 0
                matched_t.add(r)
                matched_d.add(c)

    # ── 2. Ko'rinmagan track larni aging ────────────────────────────
    for i, t in enumerate(tracks):
        if i not in matched_t:
            t.age += 1

    # ── 3. Yangi detection lar uchun Re-ID yoki yangi track ─────────
    for j, d in enumerate(dets):
        if j in matched_d:
            continue

        # Re-ID: Yo'qolgan track larning embeddingini yangi detection bilan solishtir
        reid_done = False
        crop = get_muzzle_crop(frame, d[0])
        new_emb = get_embedding(crop)

        if new_emb is not None:
            best_score = REID_THRESHOLD
            best_track_i = -1
            for i, t in enumerate(tracks):
                if i in matched_t:
                    continue  # Allaqachon matched
                if t.last_emb is None:
                    continue
                if t.state == 'tentative':
                    continue
                sim = float(np.dot(t.last_emb, new_emb))
                if sim > best_score:
                    best_score = sim
                    best_track_i = i

            if best_track_i >= 0:
                # Track qayta topildi!
                t = tracks[best_track_i]
                t.update_velocity(d[0])
                t.bbox    = d[0]
                t.conf    = d[1]
                t.hits   += 1
                t.age     = 0
                t.last_emb = new_emb
                matched_t.add(best_track_i)
                reid_done = True

        if not reid_done:
            nt = Track(d[0], d[1])
            if new_emb is not None:
                nt.last_emb = new_emb
            tracks.append(nt)

    # ── 4. Holatni yangilash ────────────────────────────────────────
    for t in tracks:
        if t.state == 'tentative' and t.hits >= HITS_TO_CONFIRM:
            t.state = 'unidentified'

    # ── 5. Identifikatsiya qilish ───────────────────────────────────
    for t in tracks:
        if t.state != 'unidentified':
            continue
        if t.animal_id is not None:
            continue
        if t.id_attempts >= MAX_ID_ATTEMPTS:
            continue
        if fno % ID_ATTEMPT_INTERVAL != 0:
            continue

        t.id_attempts += 1
        crop = get_muzzle_crop(frame, t.bbox)
        emb  = get_embedding(crop)

        if emb is not None:
            t.last_emb = emb  # Har doim yangilash (Re-ID uchun)
            aid, sc = identify(emb)
            if aid is not None:
                t.animal_id = aid
                t.id_score  = sc
                t.state     = 'identified'
                idx = AVG_ANIMAL_IDS.tolist().index(aid)
                t.tag_id = AVG_TAG_IDS[idx]

    # ── 6. Eski track larni o'chirish ───────────────────────────────
    tracks = [t for t in tracks if t.age < MAX_AGE_FRAMES]

    return tracks


# ── Bbox chizish ────────────────────────────────────────────────────
def draw_tracks(frame, tracks):
    out = frame.copy()
    H, W = out.shape[:2]
    for t in tracks:
        cx, cy, bw, bh = t.bbox
        x1 = max(0, int((cx - bw/2) * W))
        y1 = max(0, int((cy - bh/2) * H))
        x2 = min(W, int((cx + bw/2) * W))
        y2 = min(H, int((cy + bh/2) * H))
        col = COLOR[t.state]
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 2)
        lbl = f'#{t.id} {t.tag_id or "?"}'
        if t.id_score > 0:
            lbl += f' {t.id_score:.2f}'
        cv2.putText(out, lbl, (x1, max(y1-8, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        # Ko'rinmay tursa — chiziq rangi boshqacha
        if t.age > 0:
            cv2.rectangle(out, (x1, y1), (x2, y2), (128, 128, 0), 1)
    return out


# ══════════════════════════════════════════════════════════════════════
# ASOSIY LOOP
# ══════════════════════════════════════════════════════════════════════
Track._counter = 0
tracks = []

cap  = cv2.VideoCapture(VIDEO_FILE)
fps_ = cap.get(cv2.CAP_PROP_FPS) or 25
W_   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = None
if SAVE_OUTPUT_VIDEO:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(OUTPUT_VIDEO_FILE, fourcc, fps_, (W_, H_))

batch_tracks = []
fno = 0
t_start = time.time()

print('Video qayta ishlanmoqda...')
print(f'Tracker parametrlari: IoU>={IOU_MATCH_THRESHOLD} | MaxAge={MAX_AGE_FRAMES} | ReID>={REID_THRESHOLD}')

while True:
    ok, frame = cap.read()
    if not ok:
        break

    fno += 1

    # YOLO — sigirlarni topish
    results = yolo_model(frame, classes=[19], verbose=False, conf=0.35)
    dets = []
    if results and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            b    = box.xywhn[0].cpu().numpy()
            conf = float(box.conf[0])
            dets.append(((b[0], b[1], b[2], b[3]), conf))

    # Tracker
    active_tracks = step_tracker(dets, frame, fno)

    # DB ga yuborish
    for t in active_tracks:
        if t.state == 'tentative':
            continue
        batch_tracks.append({
            'track_id':     t.id,
            'animal_id':    t.animal_id,
            'tag_id':       t.tag_id,
            'state':        t.state,
            'confidence':   round(t.conf, 4),
            'id_score':     round(t.id_score, 4),
            'bbox':         {'x': round(t.bbox[0], 4), 'y': round(t.bbox[1], 4),
                             'w': round(t.bbox[2], 4), 'h': round(t.bbox[3], 4)},
            'frame_number': fno,
            'video_time_s': round(fno / fps_, 2),
        })

    if fno % PUSH_EVERY_N == 0 and batch_tracks:
        payload = {
            'camera_id':    CAMERA_ID,
            'video_file':   os.path.basename(VIDEO_FILE),
            'inference_ms': 0,
            'tracks':       batch_tracks,
        }
        try:
            requests.post(f'{BACKEND_URL}/api/v1/colab/push-tracks',
                          json=payload, headers=HEADERS, timeout=5)
        except:
            pass
        batch_tracks = []

    # Video yozish
    drawn = draw_tracks(frame, active_tracks)
    if writer:
        writer.write(drawn)

    # Preview (har 30 kadrda)
    if fno % 30 == 0:
        elapsed = time.time() - t_start
        speed   = fno / elapsed
        eta     = (TOTAL_FRAMES - fno) / max(speed, 1)
        id_cnt  = sum(1 for t in active_tracks if t.state == 'identified')
        unk_cnt = sum(1 for t in active_tracks if t.state == 'unidentified')

        clear_output(wait=True)
        _, buf = cv2.imencode('.jpg', drawn, [cv2.IMWRITE_JPEG_QUALITY, 70])
        display(IPImage(data=buf.tobytes()))
        print(f'Kadr {fno}/{TOTAL_FRAMES} | {speed:.1f} fps | ETA: {eta/60:.1f} min')
        print(f'Tracklar: {len(active_tracks)} | Aniqlandi: {id_cnt} | Noma\'lum: {unk_cnt}')

cap.release()
if writer:
    writer.release()

# Qolgan batch
if batch_tracks:
    try:
        requests.post(f'{BACKEND_URL}/api/v1/colab/push-tracks',
                      json={'camera_id': CAMERA_ID, 'video_file': os.path.basename(VIDEO_FILE),
                            'inference_ms': 0, 'tracks': batch_tracks},
                      headers=HEADERS, timeout=10)
    except:
        pass

total_time = time.time() - t_start
print(f'\n✅ Tayyor! {fno} kadr — {total_time/60:.1f} daqiqa')
if SAVE_OUTPUT_VIDEO:
    sz = os.path.getsize(OUTPUT_VIDEO_FILE)/1024/1024
    print(f'Video: {OUTPUT_VIDEO_FILE} ({sz:.1f} MB)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Chiqish videoni yuklab olish
# ═══════════════════════════════════════════════════════════════════
import os
from google.colab import files

if os.path.exists(OUTPUT_VIDEO_FILE):
    sz = os.path.getsize(OUTPUT_VIDEO_FILE)/1024/1024
    print(f'✅ Video: {OUTPUT_VIDEO_FILE} ({sz:.1f} MB)')
    files.download(OUTPUT_VIDEO_FILE)
else:
    print('❌ Video topilmadi. CELL 6 tugallandimi?')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Natijalarni tekshirish
# ═══════════════════════════════════════════════════════════════════
import requests

r = requests.get(f'{BACKEND_URL}/api/v1/animals/', headers=HEADERS)
if r.status_code == 200:
    animals = r.json()
    print(f'Jonivorlar: {len(animals)} ta')
    for a in animals[:15]:
        tag = a.get('tag_id', '?')
        name = a.get('name', '')
        last = a.get('last_detected_at', 'hech qachon')
        print(f'  [{tag}] {name} — oxirgi ko\'rinish: {last}')
else:
    print(f'❌ {r.status_code}: {r.text[:200]}')